# Improving Clinical Reasoning with AI : Lessons from 400 Kenyan Case Vignettes

## Business Understanding
In Kenya, frontline nurses often face high-stakes clinical decisions under intense pressure and with limited resources. These healthcare workers operate in environments where specialist support is scarce, yet their judgment can mean the difference between life and death.
The rise of large language models (LLMs) such as GPT-4, LLAMA, and GEMINI offers a potential support system for healthcare workers—either by providing second opinions or pre-screening suggestions. However, these systems must first be proven to emulate the reasoning and decision-making patterns of real, trained clinicians.
This project explores whether AI models can replicate or assist clinical decision-making in real Kenyan medical contexts.

## Problem Statement

1. Rural Kenyan healthcare workers make critical decisions with limited resources
2. 	Nurses across different counties and facility levels face complex medical situations daily
3. 	Need for AI that can match human clinical reasoning in low-resource settings

## Objectives
The goal is to train a model that can predict the clinician’s response to each complex clinical prompt, effectively mimicking the decision-making of trained healthcare professionals.


1. Build an AI model that can replicate clinical reasoning of human healthcare professionals.
2. Compare predictions from  models with outputs from existing LLMs (GPT-4, LLAMA, GEMINI).
3. Measure response similarity between predicted vs. real clinician answers using semantic metrics (e.g., BERTScore, BLEU, ROUGE).

4. Evaluate factual accuracy of responses based on DDX SNOMED codes (clinical diagnosis correctness).

5. Explore impact of nurse metadata (e.g., experience level, health facility) on model accuracy.

In [1]:
#Initialize libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import re # Import re for regular expressions
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('vader_lexicon', quiet=True)
except Exception as e:
    print(f"NLTK download failed: {e}")

In [2]:
#Load all the datasets
train_raw = pd.read_csv('data/train_raw.csv')
train_processed = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test_raw.csv')
test_processed = pd.read_csv('data/test.csv')


#For EDA and data understanding, we will use the train_raw.csv
df_original = pd.read_csv("data/train_raw.csv")
df_original.head()

,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...
2,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder)\n25458004 |...
3,ID_QOQTK,Uasin Gishu,National Referral Hospitals,12.0,I am a nurse with 12 years of experience in Pr...,Critical Care,INTERNAL MEDICINE,SUMMARY\n\n72-year-old female with inability t...,"Given ER's clinical presentation and vitals, t...",to me with this query. Based on the informatio...,This 92-year-old female patient (ER) presents ...,14760008 | Constipation (finding)\n419284004 |...
4,ID_ZFJBM,Uasin Gishu,National Referral Hospitals,16.0,I am a nurse with 16 years of experience in Ge...,Adult Health,INTERNAL MEDICINE,"A 22 year old female presents with headache, d...",The 22-year-old female patient is presenting w...,Thank you for presenting this case. Based on t...,This 22-year-old female patient presents with ...,95874006 | Carbon monoxide poisoning from fire...


## Data Understanding

In [3]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  300 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


The Data Dictionary is as follows:

<figure>
    <img src="pictures/data_dictionary.png" alt="Description" width="1500">
    <figcaption>Data Dictionary</figcaption>
</figure>

The data set has 12 columns. Some preliminary observations:
* There is missing data on the Years of Experience Column; However, the Prompt column seems to specify the nurses' years of experience and so we can attempt to extract missing data from here.
* The DDX SNOMED column can contain multiple diagnosis. For analytical purposes, we will need to explode the diagnoses
* Each DDX SNOMED entry appears to pack 3 levels of data i.e. a DDX code, its associated description and its diagnosis type. We will need to split it into these 3 levels for analytical purposes.

### Basic Data Structure Analysis

In [4]:
if train_raw is not None:


    print("\n" + "=" * 60)
    print("DATASET STRUCTURE OVERVIEW")
    print("=" * 60)

    datasets = {
        'Train Raw': train_raw,
        'Train Processed': train_processed,
        'Test Raw': test_raw,
        'Test Processed': test_processed
    }

    # Compare dataset shapes
    print("\n Dataset Dimensions:")
    for name, df in datasets.items():
        print(f"{name:15}: {df.shape[0]:3d} rows × {df.shape[1]:2d} columns")

    # Check columns in each dataset
    print("\n Column Comparison:")
    for name, df in datasets.items():
        print(f"\n{name} Columns ({len(df.columns)}):")
        print(df.columns.tolist())


DATASET STRUCTURE OVERVIEW

 Dataset Dimensions:
Train Raw      : 400 rows × 12 columns
Train Processed: 400 rows × 12 columns
Test Raw       : 100 rows ×  7 columns
Test Processed : 100 rows ×  7 columns

 Column Comparison:

Train Raw Columns (12):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI', 'DDX SNOMED']

Train Processed Columns (12):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI', 'DDX SNOMED']

Test Raw Columns (7):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel']

Test Processed Columns (7):
['Master_Index', 'County', 'Health level', 'Years of Experience', 'Prompt', 'Nursing Competency', 'Clinical Panel']


### Tableau  EDA

Exploratory Data Analysis was done using Tableau tools

<figure>
    <img src="pictures/Medical_Categories_Distribution.png" alt="Description" width="1500">
    <figcaption>Medical Categories Distribution</figcaption>
</figure>

Internal Medicine and Surgery comprise 50% of the cases in these hospitals.

<figure>
    <img src="pictures/Data_Distribution_By_County.png" alt="Description" width="1500">
    <figcaption>MData Distribution By County</figcaption>
</figure>

Majority of the data was collected from medical facilities in Uasin Gishu county

<figure>
    <img src="pictures/Years_of_Experience_Statistical_Summary_by_County.png" alt="Description" width="700">
    <figcaption>Years of Experience Statistical Summary by County</figcaption>
</figure>

Uasin Gishu county generally has more experience nurses

<figure>
    <img src="pictures/Years_of_Experience_Statistical_Summary_by_County_by_facility_type.png" alt="Description" width="1500">
    <figcaption>Years of Experience Statistical Summary by County and Facility Type</figcaption>
</figure>

Uasin Gishu Health Centres are the Medical Facility type with the highest Nurse Experience

<figure>
    <img src="pictures/Nurses_Average_Years_of_Experience_Per_County_Per_Facility_Type.png" alt="Description" width="1500">
    <figcaption>Nurses' Average Years of Experience Per County Per Facility Type</figcaption>
</figure>

The regional average number of years of experience by medical facility type can be used to impute missing Years of Experience data

<figure>
    <img src="pictures/Nurses_Median_Years_of_Experience_Per_County_Per_Facility_Type.png" alt="Description" width="1500">
    <figcaption>Nurses' Median Years of Experience Per County Per Facility Type</figcaption>
</figure>

The regional median number of years of experience by medical facility type can be used to impute missing Years of Experience data

<figure>
    <img src="pictures/Data_Distribution_by_Nursing_Competency.png" alt="Description" width="1500">
    <figcaption>Data Distribution by Nursing Competency</figcaption>
</figure>

Adult Health and General Emergency cases comprise approximately 50% of the cases in these regions

A few summaries from the data are shown below

In [5]:
if train_raw is not None:
    print("\n" + "=" * 60)
    print("CATEGORICAL FEATURES ANALYSIS")
    print("=" * 60)

    categorical_cols = ['County', 'Health level', 'Nursing Competency', 'Clinical Panel']

    for col in categorical_cols:
        if col in train_raw.columns:
            print(f"\n {col}:")
            value_counts = train_raw[col].value_counts()
            print(f"  Total unique values: {len(value_counts)}")
            print("  Top 5 categories:")
            for val, count in value_counts.head().items():
                percentage = (count / len(train_raw)) * 100
                print(f"    {val}: {count} ({percentage:.1f}%)")


CATEGORICAL FEATURES ANALYSIS

 County:
  Total unique values: 5
  Top 5 categories:
    Uasin Gishu: 247 (61.8%)
    Kakamega: 83 (20.8%)
    Kiambu: 60 (15.0%)
    Elgeiyo Marakwet: 6 (1.5%)
    Bungoma: 4 (1.0%)

 Health level:
  Total unique values: 8
  Top 5 categories:
    Sub-county Hospitals and Nursing Homes: 131 (32.8%)
    National Referral Hospitals: 125 (31.2%)
    Health centres: 69 (17.2%)
    Dispensaries and Private Clinics: 54 (13.5%)
    County Hospitals: 9 (2.2%)

 Nursing Competency:
  Total unique values: 21
  Top 5 categories:
    Adult Health: 123 (30.8%)
    General Emergency: 66 (16.5%)
    Child Health: 56 (14.0%)
    Maternal and Child Health: 50 (12.5%)
    Sexual And Reproductive Health: 25 (6.2%)

 Clinical Panel:
  Total unique values: 14
  Top 5 categories:
    INTERNAL MEDICINE: 133 (33.2%)
    SURGERY: 91 (22.8%)
    PAEDIATRICS: 78 (19.5%)
    OBSTETRICS AND GYNAECOLOGY: 68 (17.0%)
    CRITICAL CARE: 18 (4.5%)


### Text Data Structure Analysis

In [6]:
if train_raw is not None:

    print("\n" + "=" * 60)
    print("TEXT DATA ANALYSIS")
    print("=" * 60)

    text_columns = ['Prompt', 'Clinician', 'GPT4.0', 'LLAMA', 'GEMINI']

    for col in text_columns:
        if col in train_raw.columns:
            print(f"\n {col} Text Statistics:")

            # Calculate text lengths
            text_lengths = train_raw[col].astype(str).str.len()
            word_counts = train_raw[col].astype(str).str.split().str.len()

            print(f"  Character length - Mean: {text_lengths.mean():.0f}, Median: {text_lengths.median():.0f}")
            print(f"  Word count - Mean: {word_counts.mean():.0f}, Median: {word_counts.median():.0f}")
            print(f"  Min length: {text_lengths.min()}, Max length: {text_lengths.max()}")


TEXT DATA ANALYSIS

 Prompt Text Statistics:
  Character length - Mean: 556, Median: 542
  Word count - Mean: 96, Median: 92
  Min length: 229, Max length: 1403

 Clinician Text Statistics:
  Character length - Mean: 722, Median: 676
  Word count - Mean: 108, Median: 102
  Min length: 155, Max length: 2143

 GPT4.0 Text Statistics:
  Character length - Mean: 5419, Median: 5384
  Word count - Mean: 775, Median: 780
  Min length: 1278, Max length: 17451

 LLAMA Text Statistics:
  Character length - Mean: 2324, Median: 2308
  Word count - Mean: 335, Median: 331
  Min length: 1060, Max length: 3501

 GEMINI Text Statistics:
  Character length - Mean: 3857, Median: 3970
  Word count - Mean: 535, Median: 554
  Min length: 1071, Max length: 5576


###   Clinician vs NLP Responses Text Summary

| **Model**         | **Avg Characters** | **Avg Words** | **Style Implication**                 |
|------------------|-------------------|---------------|---------------------------------------|
| **Human Clinician** | 722               | 108           | Concise, focused                      |
| **LLAMA**         | 2,324             | 335           | ~3× longer than human                 |
| **GEMINI**        | 3,857             | 535           | ~5× longer than human                 |
| **GPT-4**         | 5,419             | 775           | ~7× longer than human                 |

---

**Critical Insight**:Human clinicians are significantly more concise than AI models.
To improve clinical utility, the model needs to **learn to be brief and focused**—not overly verbose.

### Sample Data Inspection

In [7]:
if train_raw is not None:

    print("\n" + "=" * 60)
    print("SAMPLE DATA INSPECTION")
    print("=" * 60)

    print("\n First Training Sample:")
    sample_idx = 50
    sample = train_raw.iloc[sample_idx]

    key_fields = ['Master_Index', 'County', 'Health level', 'Nursing Competency', 'Clinical Panel']
    for field in key_fields:
        if field in sample.index:
            print(f"  {field}: {sample[field]}")

    print(f"\n Prompt (truncated):")
    prompt_text = str(sample.get('Prompt', 'N/A'))
    print(f"  {prompt_text[:300]}{'...' if len(prompt_text) > 300 else ''}")

    print(f"\n Clinician Response (truncated):")
    clinician_text = str(sample.get('Clinician', 'N/A'))
    print(f"  {clinician_text[:300]}{'...' if len(clinician_text) > 300 else ''}")


SAMPLE DATA INSPECTION

 First Training Sample:
  Master_Index: ID_ICCWP
  County: Kiambu
  Health level: Sub-county Hospitals and Nursing Homes
  Nursing Competency: General Emergency
  Clinical Panel: CRITICAL CARE

 Prompt (truncated):
  I am a nurse with 12 years of experience in General nursing working in a Sub-county Hospitals and Nursing Homes in Kiambu county in Kenya. A female patient is brought to the hospital with complaint of shortness of breath upon exhaustion for three days. On examination, temperature is at 39.4, blood p...

 Clinician Response (truncated):
  SUMMARY
Female, SOB 3/7, T 39.4, BP 122/80, SpO2 80. Oxygen is empty.

How do I manage this patient?
Assess for life-threatening conditions, administer antipyretics and monitor vitals, start fluid therapy.

Perform the following laboratory tests:
 full hemogram, liver and renal functions, ESR and pr...


### Data Cleaning

#### Handle Missing Values on the Years of Experience Column
We will attempt to extract Years of Experience from the Nurses' prompts if possible

In [8]:
#Create a function that extracts level of experience from the prompt, 
def extract_experience(prompt):
    match = re.search(r"(\d+)[ ]*years of experience", prompt)
    return int(match.group(1)) if match else None

df_original['Extracted Experience'] = df_original['Prompt'].apply(extract_experience) #Extract the experience level from the nurses prompt
df_original['Experience Match'] = df_original['Extracted Experience'] == df_original['Years of Experience'] #check if the extracted experience level matches the years of experience level column and score True or False
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Master_Index          400 non-null    object 
 1   County                400 non-null    object 
 2   Health level          400 non-null    object 
 3   Years of Experience   300 non-null    float64
 4   Prompt                400 non-null    object 
 5   Nursing Competency    400 non-null    object 
 6   Clinical Panel        400 non-null    object 
 7   Clinician             400 non-null    object 
 8   GPT4.0                400 non-null    object 
 9   LLAMA                 400 non-null    object 
 10  GEMINI                400 non-null    object 
 11  DDX SNOMED            399 non-null    object 
 12  Extracted Experience  300 non-null    float64
 13  Experience Match      400 non-null    bool   
dtypes: bool(1), float64(2), object(11)
memory usage: 41.1+ KB


Even after attempting to years of experience from the prompt, there is still missing data. A manual inspection of some rows with this missing data showed that these nurses did not specify their experience level in their prompt. As experience level may be crucial in the model, we decided to impute by assigning it the mean of nurses in that county and that facility type.

We can then drop these 2 new columns as we do not need them any further

In [9]:
df_original = df_original.drop(['Extracted Experience', 'Experience Match'], axis=1)
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  300 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


Impute the missing Years of Experience using the avaerage for that county and medical facility type

In [10]:
#Impute missing Years of Experience with mean based on that county and that facility type
df_original['Years of Experience'] = df_original.apply(
    lambda row: df_original[(df_original['County'] == row['County']) & (df_original['Health level'] == row['Health level'])]['Years of Experience'].mean()
    if pd.isna(row['Years of Experience']) else row['Years of Experience'], axis=1
)
df_cleaned = df_original.copy()
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         400 non-null    object 
 1   County               400 non-null    object 
 2   Health level         400 non-null    object 
 3   Years of Experience  400 non-null    float64
 4   Prompt               400 non-null    object 
 5   Nursing Competency   400 non-null    object 
 6   Clinical Panel       400 non-null    object 
 7   Clinician            400 non-null    object 
 8   GPT4.0               400 non-null    object 
 9   LLAMA                400 non-null    object 
 10  GEMINI               400 non-null    object 
 11  DDX SNOMED           399 non-null    object 
dtypes: float64(1), object(11)
memory usage: 37.6+ KB


#### Exploding DDX SNOMED Values 
We will attempt to explode the Data to seperate multiple diagnoses into individual rows

In [11]:
df_exploded = df_cleaned.copy()
df_exploded["DDX SNOMED Split"] = df_exploded["DDX SNOMED"].str.split("\n")
df_exploded = df_exploded.explode("DDX SNOMED Split").reset_index(drop=True)
df_exploded.drop(columns=["DDX SNOMED"], inplace=True)
df_exploded.head()


,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED Split
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...
2,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",700055009 | Sepsis with cutaneous manifestatio...
3,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder)
4,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",25458004 | Acute gastritis (disorder)


In [12]:
df_exploded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Master_Index         1202 non-null   object 
 1   County               1202 non-null   object 
 2   Health level         1202 non-null   object 
 3   Years of Experience  1202 non-null   float64
 4   Prompt               1202 non-null   object 
 5   Nursing Competency   1202 non-null   object 
 6   Clinical Panel       1202 non-null   object 
 7   Clinician            1202 non-null   object 
 8   GPT4.0               1202 non-null   object 
 9   LLAMA                1202 non-null   object 
 10  GEMINI               1202 non-null   object 
 11  DDX SNOMED Split     1201 non-null   object 
dtypes: float64(1), object(11)
memory usage: 112.8+ KB


The data has now been exploded such that each diagnosis appears on its own row. Next, we need to engineer the 3 layers of packed information from the DDX SNOMED Split column into a DDX code, its description and DDX Type

In [13]:
# Function to extract Code, Description, and Type from the DDX SNOMED Column
def extract_snomed_details(snomed_split_str):
    if pd.isna(snomed_split_str):
        return pd.NA, pd.NA, pd.NA

    # Using regex to capture the code, description, and the part in parentheses
    match = re.match(r'(\d+)\s*\|\s*(.*?)\s*\((.*?)\)', snomed_split_str)

    if match:
        code = match.group(1)
        description = match.group(2).strip()
        snomed_type = match.group(3).strip()
        return code, description, snomed_type
    else:
        # Handle cases that don't match the expected format
        return pd.NA, pd.NA, pd.NA

# Apply the function to the 'DDX SNOMED Split' column
df_exploded[['DDX Code', 'DDX Description', 'DDX Type']] = df_exploded['DDX SNOMED Split'].apply(
    lambda x: pd.Series(extract_snomed_details(x))
)

df_exploded.head()

,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED Split,DDX Code,DDX Description,DDX Type
0,ID_VBWWP,Uasin Gishu,Sub-county Hospitals and Nursing Homes,18.0,I am a nurse with 18 years of experience in Ge...,Pediatric Emergency Burns,SURGERY,Summary:\nA 4 year old with 5% superficial bur...,Given your vast experience as a nurse in Uasin...,1. Immediate treatment protocol for second-deg...,Here's a response addressing the questions reg...,288514009 | Burn involving 5 percent of body s...,288514009,Burn involving 5 percent of body surface,disorder
1,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",420270002 | Ketoacidosis due to type 1 diabete...,420270002,Ketoacidosis due to type 1 diabetes mellitus,disorder
2,ID_XMBBY,Uasin Gishu,National Referral Hospitals,17.0,I am a nurse with 17 years of experience in Ge...,Child Health,PAEDIATRICS,Summary\n6-year-old present with vomiting and ...,Clinical Summary: • A 6-year-old girl w...,Based on the symptoms and signs you've describ...,"Based on the presentation, the 6-year-old girl...",700055009 | Sepsis with cutaneous manifestatio...,700055009,Sepsis with cutaneous manifestations,disorder
3,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",13200003 | Peptic ulcer (disorder),13200003,Peptic ulcer,disorder
4,ID_JZNZW,Kiambu,Sub-county Hospitals and Nursing Homes,12.0,I am a nurse with 12 years of experience in Ge...,General Emergency,INTERNAL MEDICINE,Summary\nA 47-year-old man presents with sever...,"In this case, you're dealing with a 47-year-ol...","Firstly, I must commend you on your thorough h...","This 47-year-old male presenting with severe, ...",25458004 | Acute gastritis (disorder),25458004,Acute gastritis,disorder


With this clean exploded data, we can now analyze the data further e.g. frequency of certain diagnoses, distribution of diagnoses per county or per facility type (or both) etc.
Later, we could attempt extracting sentiments from the prompts that match with the descriptions from the DDX SNOMED.

In [14]:
df_exploded.to_csv("data/df_exploded.csv", index=False)

## Clinical Reasoning - ML Pipeline
### Key Features of This Pipeline:

#### Smart Feature Engineering

1. **Patient demographics**: Extracts age, gender from prompts
2. **Clinical urgency**: Scores based on emergency keywords
3. **Medical complexity**: Counts medical terminology
4. **Experience integration**: Uses the "Years of Experience" column
5. **Categorical handling**: Manages unseen categories in test data

#### Distribution Shift Solution

1. Label encoding with fallback: Handles unseen nursing competencies/clinical panels
2. TF-IDF focus: Relies more on prompt content than metadata
3. Robust preprocessing: Won't break on new categories

#### Response Length Awareness

1. Target length prediction: Models learn appropriate response length
2. Template-based generation: Uses clinical specialty-specific templates
3. Conciseness focus: Targets human-style brevity (not AI verbosity)

#### Clinical Context Integration

1. Specialty-specific templates: Different formats for Surgery vs Pediatrics
2. Urgency adaptation: Emergency cases get different language
3. Resource awareness: Considers facility level in responses

### Extract structured features from clinical prompts and metadata

In [15]:
class ClinicalFeatureExtractor:
    """Extract structured features from clinical prompts and metadata"""

    def __init__(self):
        self.medical_terms = [
            'fever', 'pain', 'bleeding', 'pregnant', 'delivery', 'emergency',
            'surgery', 'medication', 'treatment', 'diagnosis', 'patient',
            'symptoms', 'condition', 'severe', 'acute', 'chronic', 'weeks',
            'months', 'years', 'old', 'male', 'female', 'child', 'adult'
        ]

        self.urgency_indicators = [
            'emergency', 'urgent', 'critical', 'severe', 'acute', 'bleeding',
            'unconscious', 'shock', 'arrest', 'trauma', 'accident'
        ]

    def extract_patient_age(self, prompt):
        """Extract patient age from prompt text"""
        age_patterns = [
            r'(\d+)\s*(?:year|yr)s?\s*old',
            r'(\d+)\s*(?:month|mo)s?\s*old',
            r'(\d+)\s*(?:week|wk)s?\s*old',
            r'age\s*(\d+)',
            r'(\d+)\s*y/?o'
        ]

        for pattern in age_patterns:
            match = re.search(pattern, prompt.lower())
            if match:
                age = int(match.group(1))
                # Convert to years if needed
                if 'month' in pattern or 'mo' in pattern:
                    age = age / 12
                elif 'week' in pattern or 'wk' in pattern:
                    age = age / 52
                return age
        return None

    def extract_patient_gender(self, prompt):
        """Extract patient gender from prompt"""
        prompt_lower = prompt.lower()

        male_indicators = ['male', 'man', 'boy', 'father', 'husband', 'his', 'he ']
        female_indicators = ['female', 'woman', 'girl', 'mother', 'wife', 'her', 'she ']

        male_count = sum(1 for term in male_indicators if term in prompt_lower)
        female_count = sum(1 for term in female_indicators if term in prompt_lower)

        if male_count > female_count:
            return 'male'
        elif female_count > male_count:
            return 'female'
        return 'unknown'

    def extract_urgency_score(self, prompt):
        """Calculate urgency score based on keywords"""
        prompt_lower = prompt.lower()
        urgency_score = 0

        for term in self.urgency_indicators:
            if term in prompt_lower:
                urgency_score += 1

        return urgency_score

    def extract_medical_complexity(self, prompt):
        """Calculate medical complexity score"""
        prompt_lower = prompt.lower()
        complexity_score = 0

        for term in self.medical_terms:
            complexity_score += prompt_lower.count(term)

        return complexity_score

    def extract_prompt_features(self, df):
        """Extract all features from prompts"""
        features = pd.DataFrame()

        print(" Extracting features from prompts...")

        # Basic text features
        features['prompt_length'] = df['Prompt'].str.len()
        features['prompt_word_count'] = df['Prompt'].str.split().str.len()
        features['prompt_sentence_count'] = df['Prompt'].apply(
            lambda x: len(sent_tokenize(x)) if x else 0
        )

        # Patient features - using self. to call instance methods
        features['patient_age'] = df['Prompt'].apply(self.extract_patient_age)
        features['patient_gender'] = df['Prompt'].apply(self.extract_patient_gender)
        features['urgency_score'] = df['Prompt'].apply(self.extract_urgency_score)
        features['medical_complexity'] = df['Prompt'].apply(self.extract_medical_complexity)

        # Fill missing ages with median
        median_age = features['patient_age'].median()
        features['patient_age'] = features['patient_age'].fillna(median_age)

        # Age categories
        features['age_category'] = pd.cut(
            features['patient_age'],
            bins=[0, 1, 12, 18, 65, 100],
            labels=['infant', 'child', 'adolescent', 'adult', 'elderly'],
            include_lowest=True
        )

        print(f" Extracted {len(features.columns)} prompt-based features")
        return features

### Prepare all features for modeling

In [16]:
def prepare_features(train_df, test_df):
    """Prepare all features for modeling"""

    print("\n FEATURE ENGINEERING")
    print("=" * 50)

    # Initialize feature extractor
    extractor = ClinicalFeatureExtractor()

    # Extract prompt features
    train_prompt_features = extractor.extract_prompt_features(train_df)
    test_prompt_features = extractor.extract_prompt_features(test_df)

    # Combine with existing features
    train_features = pd.concat([
        train_df[['Master_Index', 'County', 'Health level', 'Years of Experience',
                 'Nursing Competency', 'Clinical Panel', 'Prompt']].copy(),
        train_prompt_features
    ], axis=1)

    test_features = pd.concat([
        test_df[['Master_Index', 'County', 'Health level', 'Years of Experience',
                'Nursing Competency', 'Clinical Panel', 'Prompt']].copy(),
        test_prompt_features
    ], axis=1)

    # Handle categorical variables with potential unseen categories
    categorical_cols = ['County', 'Health level', 'Nursing Competency', 'Clinical Panel',
                       'patient_gender', 'age_category']

    print(f"\n Processing categorical features...")

    # Use label encoding with unknown category handling
    encoders = {}
    for col in categorical_cols:
        if col in train_features.columns:
            encoder = LabelEncoder()

            # Fit on training data
            train_values = train_features[col].astype(str).fillna('unknown')
            encoder.fit(train_values)

            # Transform training data
            train_features[f'{col}_encoded'] = encoder.transform(train_values)

            # Transform test data, handling unseen categories
            test_values = test_features[col].astype(str).fillna('unknown')
            test_encoded = []

            for val in test_values:
                if val in encoder.classes_:
                    test_encoded.append(encoder.transform([val])[0])
                else:
                    # Assign to most common class for unseen categories
                    most_common_class = train_features[col].value_counts().index[0]
                    test_encoded.append(encoder.transform([most_common_class])[0])

            test_features[f'{col}_encoded'] = test_encoded
            encoders[col] = encoder

    print(f" Processed {len(categorical_cols)} categorical features")

    return train_features, test_features, encoders

### TEXT VECTORIZATION

In [17]:
def create_text_features(train_df, test_df, max_features=5000):
    """Create TF-IDF features from prompts"""

    print(f"\n TEXT VECTORIZATION")
    print("=" * 50)

    # Initialize TF-IDF vectorizer
    tfidf = TfidfVectorizer(
        max_features=max_features,
        stop_words='english',
        ngram_range=(1, 2),  # Include bigrams
        min_df=2,  # Ignore terms that appear in less than 2 documents
        max_df=0.95  # Ignore terms that appear in more than 95% of documents
    )

    # Fit on training prompts
    train_prompts = train_df['Prompt'].fillna('')
    test_prompts = test_df['Prompt'].fillna('')

    # Fit and transform training data
    train_tfidf = tfidf.fit_transform(train_prompts)
    test_tfidf = tfidf.transform(test_prompts)

    print(f" Created TF-IDF features: {train_tfidf.shape[1]} dimensions")

    return train_tfidf, test_tfidf, tfidf

### RESPONSE ANALYSIS & TARGET PREPARATION

In [18]:
def analyze_target_variable(train_df):
    """Analyze the target variable (Clinician responses)"""

    print(f"\n TARGET VARIABLE ANALYSIS")
    print("=" * 50)

    clinician_responses = train_df['Clinician'].fillna('')

    # Response length statistics
    response_lengths = clinician_responses.str.len()
    response_word_counts = clinician_responses.str.split().str.len()

    print(f"Response Length Statistics:")
    print(f"  Mean: {response_lengths.mean():.0f} characters")
    print(f"  Median: {response_lengths.median():.0f} characters")
    print(f"  Std: {response_lengths.std():.0f} characters")
    print(f"  Min: {response_lengths.min()}, Max: {response_lengths.max()}")

    print(f"\nWord Count Statistics:")
    print(f"  Mean: {response_word_counts.mean():.0f} words")
    print(f"  Median: {response_word_counts.median():.0f} words")

    # Common response patterns
    summary_count = sum(1 for resp in clinician_responses if 'summary' in resp.lower())
    diagnosis_count = sum(1 for resp in clinician_responses if 'diagnosis' in resp.lower())
    management_count = sum(1 for resp in clinician_responses if 'management' in resp.lower())

    print(f"\nCommon Response Patterns:")
    print(f"  Contains 'Summary': {summary_count} ({summary_count/len(clinician_responses)*100:.1f}%)")
    print(f"  Contains 'Diagnosis': {diagnosis_count} ({diagnosis_count/len(clinician_responses)*100:.1f}%)")
    print(f"  Contains 'Management': {management_count} ({management_count/len(clinician_responses)*100:.1f}%)")

    return response_lengths, response_word_counts

## BASELINE MODELS

### Baseline model for predicting clinical responses

In [19]:
class ClinicalResponsePredictor:


    def __init__(self):
        self.models = {}
        self.feature_names = []
        self.tfidf_vectorizer = None
        self.scaler = StandardScaler()

    def prepare_model_features(self, structured_features, tfidf_features):
        """Combine structured and text features"""

        # Select numerical features
        numerical_cols = [
            'Years of Experience', 'prompt_length', 'prompt_word_count',
            'prompt_sentence_count', 'patient_age', 'urgency_score', 'medical_complexity'
        ]

        categorical_encoded_cols = [col for col in structured_features.columns
                                  if col.endswith('_encoded')]

        # Combine features
        feature_cols = numerical_cols + categorical_encoded_cols
        available_cols = [col for col in feature_cols if col in structured_features.columns]

        structured_array = structured_features[available_cols].fillna(0).values
        tfidf_array = tfidf_features.toarray()

        # Combine features
        combined_features = np.hstack([structured_array, tfidf_array])

        self.feature_names = available_cols + [f'tfidf_{i}' for i in range(tfidf_array.shape[1])]

        return combined_features

    def fit(self, X, y):
        """Train multiple baseline models"""

        print(f"\n TRAINING BASELINE MODELS")
        print("=" * 50)

        # Scale features
        X_scaled = self.scaler.fit_transform(X)

        # For text generation, we'll use a simple approach:
        # Create numerical targets from response characteristics
        y_lengths = [len(str(response)) for response in y]
        y_word_counts = [len(str(response).split()) for response in y]

        # Train models to predict response characteristics
        self.models['length_predictor'] = Ridge(alpha=1.0)
        self.models['word_count_predictor'] = Ridge(alpha=1.0)

        self.models['length_predictor'].fit(X_scaled, y_lengths)
        self.models['word_count_predictor'].fit(X_scaled, y_word_counts)

        # Calculate baseline scores
        length_score = self.models['length_predictor'].score(X_scaled, y_lengths)
        word_count_score = self.models['word_count_predictor'].score(X_scaled, y_word_counts)

        print(f" Length Prediction R² Score: {length_score:.3f}")
        print(f" Word Count Prediction R² Score: {word_count_score:.3f}")

        return self

    def generate_response_template(self, predicted_length, predicted_word_count,
                                 prompt, clinical_panel, nursing_competency):
        """Generate a template response based on predictions and context"""

        # Common response templates by clinical panel
        templates = {
            'INTERNAL MEDICINE': "Summary: {summary}\nDIAGNOSIS: {diagnosis}\nMANAGEMENT: {management}",
            'SURGERY': "Summary: {summary}\nIMMEDIATE MANAGEMENT: {immediate}\nSURGICAL PLAN: {surgical}",
            'PAEDIATRICS': "Summary: {summary}\nASSESSMENT: {assessment}\nMANAGEMENT: {management}",
            'OBSTETRICS AND GYNAECOLOGY': "Summary: {summary}\nDIAGNOSIS: {diagnosis}\nPLAN: {plan}",
            'CRITICAL CARE': "Summary: {summary}\nIMMEDIATE ACTIONS: {immediate}\nMONITORING: {monitoring}"
        }

        # Default template
        default_template = "Summary: {summary}\nASSESSMENT: {assessment}\nMANAGEMENT: {management}"

        template = templates.get(clinical_panel, default_template)

        # Extract key information from prompt for template filling
        prompt_lower = prompt.lower()

        # Simple keyword-based content generation
        summary_content = "Clinical case as described"
        assessment_content = "Further evaluation needed"
        management_content = "Appropriate clinical management as indicated"

        if 'emergency' in prompt_lower or 'urgent' in prompt_lower:
            management_content = "Immediate medical attention required"

        if 'child' in prompt_lower or 'pediatric' in prompt_lower:
            assessment_content = "Pediatric assessment required"

        # Fill template
        response = template.format(
            summary=summary_content,
            diagnosis=assessment_content,
            management=management_content,
            assessment=assessment_content,
            plan=management_content,
            immediate=management_content,
            surgical="Surgical consultation as needed",
            monitoring="Continue monitoring patient status"
        )

        # Adjust length to match prediction
        words = response.split()
        target_words = max(50, min(200, int(predicted_word_count)))  # Reasonable bounds

        if len(words) > target_words:
            response = ' '.join(words[:target_words])
        elif len(words) < target_words * 0.8:
            # Add some standard medical advice
            response += " Continue supportive care and monitor patient progress."

        return response

    def predict(self, X, structured_features):
        """Generate response predictions"""

        print(f"\n GENERATING PREDICTIONS")
        print("=" * 50)

        X_scaled = self.scaler.transform(X)

        # Predict response characteristics
        predicted_lengths = self.models['length_predictor'].predict(X_scaled)
        predicted_word_counts = self.models['word_count_predictor'].predict(X_scaled)

        # Generate responses
        predictions = []

        for i in range(len(X_scaled)):
            response = self.generate_response_template(
                predicted_lengths[i],
                predicted_word_counts[i],
                structured_features.iloc[i]['Prompt'],
                structured_features.iloc[i]['Clinical Panel'],
                structured_features.iloc[i]['Nursing Competency']
            )
            predictions.append(response)

        print(f" Generated {len(predictions)} response predictions")
        return predictions

## MAIN PIPELINE EXECUTION

### Execute the complete ML pipeline

In [20]:
def main_pipeline():
    """Execute the complete ML pipeline"""

    # Load data
    print(" Loading datasets...")
    train_df = pd.read_csv('data/train_raw.csv')
    test_df = pd.read_csv('data/test_raw.csv')

    print(f" Loaded {len(train_df)} training samples, {len(test_df)} test samples")

    # Feature engineering
    train_features, test_features, encoders = prepare_features(train_df, test_df)

    # Text vectorization
    train_tfidf, test_tfidf, tfidf_vectorizer = create_text_features(train_df, test_df)

    # Target analysis
    response_lengths, response_word_counts = analyze_target_variable(train_df)

    # Initialize predictor
    predictor = ClinicalResponsePredictor()
    predictor.tfidf_vectorizer = tfidf_vectorizer

    # Prepare features for modeling
    X_train = predictor.prepare_model_features(train_features, train_tfidf)
    X_test = predictor.prepare_model_features(test_features, test_tfidf)

    print(f"\n Final feature dimensions: {X_train.shape}")

    # Train models
    y_train = train_df['Clinician'].fillna('')
    predictor.fit(X_train, y_train)

    # Generate predictions
    predictions = predictor.predict(X_test, test_features)

    # Create submission file
    submission = pd.DataFrame({
        'Master_Index': test_df['Master_Index'],
        'Clinician': predictions
    })

    submission.to_csv('kenya_clinical_predictions.csv', index=False)

    print(f"\n PIPELINE COMPLETE!")
    print("=" * 50)
    print(f" Predictions saved to 'kenya_clinical_predictions.csv'")
    print(f" Generated {len(predictions)} clinical response predictions")

    # Show sample predictions
    print(f"\n Sample Predictions:")
    for i in range(min(1, len(predictions))):
        print(f"\nSample {i+1} (ID: {test_df.iloc[i]['Master_Index']}):")
        print(f"Prompt: {test_df.iloc[i]['Prompt'][:100]}...")
        print(f"Prediction: {predictions[i][:200]}...")

    return submission, predictor

# Run the pipeline
if __name__ == "__main__":
    submission, trained_predictor = main_pipeline()

 Loading datasets...
 Loaded 400 training samples, 100 test samples

 FEATURE ENGINEERING
 Extracting features from prompts...
 Extracted 8 prompt-based features
 Extracting features from prompts...
 Extracted 8 prompt-based features

 Processing categorical features...
 Processed 6 categorical features

 TEXT VECTORIZATION
 Created TF-IDF features: 3096 dimensions

 TARGET VARIABLE ANALYSIS
Response Length Statistics:
  Mean: 722 characters
  Median: 676 characters
  Std: 294 characters
  Min: 155, Max: 2143

Word Count Statistics:
  Mean: 108 words
  Median: 102 words

Common Response Patterns:
  Contains 'Summary': 325 (81.2%)
  Contains 'Diagnosis': 200 (50.0%)
  Contains 'Management': 252 (63.0%)

 Final feature dimensions: (400, 3109)

 TRAINING BASELINE MODELS
 Length Prediction R² Score: 1.000
 Word Count Prediction R² Score: 1.000

 GENERATING PREDICTIONS
 Generated 100 response predictions

 PIPELINE COMPLETE!
 Predictions saved to 'kenya_clinical_predictions.csv'
 Generated 

In [21]:
# Read the CSV file into a DataFrame
df = pd.read_csv('kenya_clinical_predictions.csv')

# Display the head of the DataFrame
df.head(10)

,Master_Index,Clinician
0,ID_CUAOY,Summary: Clinical case as described\nASSESSMEN...
1,ID_OGSAY,Summary: Clinical case as described\nIMMEDIATE...
2,ID_TYHSA,Summary: Clinical case as described\nDIAGNOSIS...
3,ID_CZXLD,Summary: Clinical case as described\nASSESSMEN...
4,ID_ZJQUQ,Summary: Clinical case as described\nASSESSMEN...
5,ID_HYSCV,Summary: Clinical case as described\nIMMEDIATE...
6,ID_DXHPF,Summary: Clinical case as described\nIMMEDIATE...
7,ID_GDFDN,Summary: Clinical case as described\nDIAGNOSIS...
8,ID_UFAFI,Summary: Clinical case as described\nDIAGNOSIS...
9,ID_KMBGG,Summary: Clinical case as described\nIMMEDIATE...


## Train an advanced NLP model
##Implement T5 Model for Response Generation
T5 (Text-to-Text Transfer Transformer)

In [22]:
# Import required libraries
import os
import gc
import re
import time
import pandas as pd
import numpy as np
import torch
import random
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from sklearn.model_selection import train_test_split, StratifiedKFold

# Hugging Face Transformers
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)

# Metrics
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Set environment variables for better performance
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# Fix random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
else:
    print("Using single GPU or CPU")

Using device: cpu
Using single GPU or CPU


In [23]:
# Load datasets
def load_data():
    """Load training and test datasets"""
    train_df = pd.read_csv('data/train.csv')
    test_df = pd.read_csv('data/test.csv')

    print(f"Training data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")

    # Display sample data
    print("\nTraining data sample:")
    display(train_df.head(2))

    return train_df, test_df

train_df, test_df = load_data()

# Preprocess text
def preprocess_text(text):
    """Basic text preprocessing for clinical text"""
    if not isinstance(text, str):
        return ""

    # Normalize spaces and newlines
    text = re.sub(r'\s+', ' ', text)

    # Preserve medical abbreviations with periods
    text = re.sub(r'([A-Za-z]\.)+([A-Za-z]\.)', lambda m: m.group().replace('.', '~DOT~'), text)

    # Keep important punctuation for medical text
    text = re.sub(r'[^\w\s.,;:%\-\/()]+', ' ', text)

    # Restore preserved abbreviations
    text = text.replace('~DOT~', '.')

    # Normalize medical measurements
    text = re.sub(r'(\d+)[\s]*(?:mg|mgs|mcg|µg|ml|mls)', lambda m: f"{m.group(1)} {m.group()[len(m.group(1)):].strip()}", text)

    # Normalize percentages
    text = re.sub(r'(\d+)[\s]*(?:percent|pct)', r'\1%', text)

    return text.strip()

# Create enhanced prompts
def create_prompt(row):
    """Create enhanced prompt with medical context"""
    prompt = row['Prompt'].strip()

    # Add context if available
    context_parts = []
    if 'Nursing Competency' in row and not pd.isna(row['Nursing Competency']):
        context_parts.append(f"Competency: {row['Nursing Competency']}")
    if 'Clinical Panel' in row and not pd.isna(row['Clinical Panel']):
        context_parts.append(f"Panel: {row['Clinical Panel']}")
    if 'Years of Experience' in row and not pd.isna(row['Years of Experience']):
        context_parts.append(f"Experience: {int(row['Years of Experience'])} yrs")

    if context_parts:
        prompt = f"Medical Context [{' | '.join(context_parts)}]: {prompt}"

    # Add patient information if found in the text
    age_gender = []
    age_match = re.search(r'(\d+)[- ]?(?:year|yr)[- ]old', prompt.lower())
    if age_match:
        age_gender.append(f"Age: {age_match.group(1)}")

    gender_match = re.search(r'\b(male|female|man|woman)\b', prompt.lower())
    if gender_match:
        gender = gender_match.group(1).replace("man", "male").replace("woman", "female")
        age_gender.append(f"Gender: {gender}")

    if age_gender:
        prompt = f"Patient [{ ' | '.join(age_gender) }] - {prompt}"

    return f"Based on clinical reasoning, provide a concise professional assessment for: {prompt}"

# Apply preprocessing
print("Preprocessing data...")
train_df['Enhanced_Prompt'] = train_df.apply(create_prompt, axis=1)
test_df['Enhanced_Prompt'] = test_df.apply(create_prompt, axis=1)

train_df['Enhanced_Prompt'] = train_df['Enhanced_Prompt'].apply(preprocess_text)
train_df['Clinician'] = train_df['Clinician'].apply(preprocess_text)
test_df['Enhanced_Prompt'] = test_df['Enhanced_Prompt'].apply(preprocess_text)

print("Sample enhanced prompt:")
print(train_df['Enhanced_Prompt'].iloc[0])
print("\nSample clinician response:")
print(train_df['Clinician'].iloc[0])

Training data shape: (400, 12)
Test data shape: (100, 7)

Training data sample:


,Master_Index,County,Health level,Years of Experience,Prompt,Nursing Competency,Clinical Panel,Clinician,GPT4.0,LLAMA,GEMINI,DDX SNOMED
0,ID_VBWWP,uasin gishu,sub county hospitals and nursing homes,18.0,i am a nurse with 18 years of experience in ge...,pediatric emergency burns,surgery,summary a 4 year old with 5 superficial burns ...,given your vast experience as a nurse in uasin...,1 immediate treatment protocol for second degr...,here s a response addressing the questions reg...,288514009 burn involving 5 percent of body sur...
1,ID_XMBBY,uasin gishu,national referral hospitals,17.0,i am a nurse with 17 years of experience in ge...,child health,paediatrics,summary 6 year old present with vomiting and a...,clinical summary • a 6 year old girl with know...,based on the symptoms and signs you ve describ...,based on the presentation the 6 year old girl ...,420270002 ketoacidosis due to type 1 diabetes ...


Preprocessing data...
Sample enhanced prompt:
Based on clinical reasoning, provide a concise professional assessment for: Patient  Age: 4  - Medical Context  Competency: pediatric emergency burns   Panel: surgery   Experience: 18 yrs : i am a nurse with 18 years of experience in general nursing working in a sub county hospitals and nursing homes in uasin gishu county in kenya a 4 year old child presents to the emergency department with second degree burns on the forearm after accidentally touching a hot stove the child was playing in the kitchen when they reached out to touch the stove the burns cover about 5 of the total body surface area the child is alert and crying with redness blisters and swelling on the affected area the burns appear to be superficial to moderate in severity the child is in mild pain and there is no indication of airway or breathing distress no other injuries are noted questions 1 what is the immediate treatment protocol for second degree burns in paediatric pat

In [24]:
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-base")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [25]:
# Define model parameters
MODEL_NAME = "t5-base"  # Options: t5-small, t5-base, t5-large
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8

# Load tokenizer and model
print(f"Loading {MODEL_NAME} model and tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Move model to device
model = model.to(device)
print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()
model.config.use_cache = False  # Disable KV cache during training

# Tokenize data efficiently
def tokenize_data(examples, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH):
    """Tokenize inputs and targets"""
    inputs = tokenizer(
        ["summarize: " + text for text in examples['Enhanced_Prompt']],
        max_length=max_source_length,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['Clinician'],
            max_length=max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

    # Replace padding token id with -100 for loss calculation
    labels_with_ignore = []
    for label in labels['input_ids']:
        labels_with_ignore.append([l if l != tokenizer.pad_token_id else -100 for l in label])

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': torch.tensor(labels_with_ignore)
    }

Loading t5-base model and tokenizer...
Model loaded with 222,903,552 parameters


In [26]:
# Create datasets for training
def prepare_datasets(df, test_size=0.2):
    """Prepare datasets for training"""
    # Create a stratified split based on clinician response length
    df['length_category'] = pd.qcut(df['Clinician'].str.len(), 4, labels=False)

    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        random_state=SEED,
        stratify=df['length_category']
    )

    print(f"Training set: {len(train_df)} examples")
    print(f"Validation set: {len(val_df)} examples")

    # Convert to datasets format
    train_dataset = Dataset.from_pandas(train_df[['Enhanced_Prompt', 'Clinician']])
    val_dataset = Dataset.from_pandas(val_df[['Enhanced_Prompt', 'Clinician']])

    # Apply tokenization
    tokenized_train = train_dataset.map(
        tokenize_data,
        batched=True,
        batch_size=16,
        remove_columns=['Enhanced_Prompt', 'Clinician'],
        desc="Tokenizing training data"
    )

    tokenized_val = val_dataset.map(
        tokenize_data,
        batched=True,
        batch_size=16,
        remove_columns=['Enhanced_Prompt', 'Clinician'],
        desc="Tokenizing validation data"
    )

    return tokenized_train, tokenized_val, train_df, val_df

# Initialize ROUGE scorer
rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Prepare test dataset
def prepare_test_dataset(test_df):
    """Prepare test dataset for inference"""
    test_dataset = Dataset.from_pandas(test_df[['Enhanced_Prompt']])

    # Tokenize test data
    tokenized_test = test_dataset.map(
        lambda examples: tokenizer(
            ["summarize: " + text for text in examples['Enhanced_Prompt']],
            max_length=MAX_SOURCE_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        ),
        batched=True,
        batch_size=16,
        remove_columns=['Enhanced_Prompt'],
        desc="Tokenizing test data"
    )

    return tokenized_test

# Prepare datasets
tokenized_train, tokenized_val, train_subset, val_subset = prepare_datasets(train_df)
tokenized_test = prepare_test_dataset(test_df)

Training set: 320 examples
Validation set: 80 examples


Tokenizing training data:   0%|          | 0/320 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing test data:   0%|          | 0/100 [00:00<?, ? examples/s]

## Compute ROUGE and other metrics for model evaluation

In [27]:
# Evaluation metrics function
def compute_metrics(eval_pred):
    """Compute ROUGE and other metrics for model evaluation"""
    predictions, labels = eval_pred

    # When using predict_with_generate=True, predictions are already the generated token sequences
    if isinstance(predictions, tuple):
        predictions = predictions[0]  # In some cases, predictions might be a tuple

    # Add safety check: filter out any token IDs outside of tokenizer's vocabulary range
    vocab_size = tokenizer.vocab_size
    # Convert to numpy and make a copy to avoid modifying the original predictions
    predictions_np = predictions.copy()
    # Replace any out-of-range IDs with the unknown token ID
    mask = (predictions_np >= vocab_size) | (predictions_np < 0)
    if mask.any():
        print(f"Warning: Found {mask.sum()} token IDs outside of vocabulary range. Replacing with <unk> token.")
        predictions_np[mask] = tokenizer.unk_token_id

    try:
        decoded_preds = tokenizer.batch_decode(predictions_np, skip_special_tokens=True)
    except Exception as e:
        print(f"Error in batch_decode: {e}")
        # Handle individual sequences with error handling
        decoded_preds = []
        for i, seq in enumerate(predictions_np):
            try:
                # Process each sequence individually to avoid failing the entire batch
                decoded = tokenizer.decode(seq, skip_special_tokens=True)
                decoded_preds.append(decoded)
            except Exception as e:
                print(f"Error decoding sequence {i}: {e}")
                decoded_preds.append("")  # Add empty string as fallback

    # Replace -100s in labels with pad token id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    try:
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f"Error decoding labels: {e}")
        # Handle individual sequences with error handling
        decoded_labels = []
        for seq in labels:
            try:
                decoded = tokenizer.decode(seq, skip_special_tokens=True)
                decoded_labels.append(decoded)
            except:
                decoded_labels.append("")

    # Post-process predictions and labels
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Calculate ROUGE scores with error handling
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, label in zip(decoded_preds, decoded_labels):
        if not pred or not label:  # Skip empty strings
            rouge1_scores.append(0.0)
            rouge2_scores.append(0.0)
            rougeL_scores.append(0.0)
            continue

        try:
            scores = rouge_scorer_obj.score(label, pred)
            rouge1_scores.append(scores['rouge1'].fmeasure)
            rouge2_scores.append(scores['rouge2'].fmeasure)
            rougeL_scores.append(scores['rougeL'].fmeasure)
        except Exception as e:
            print(f"Error calculating ROUGE scores: {e}")
            rouge1_scores.append(0.0)
            rouge2_scores.append(0.0)
            rougeL_scores.append(0.0)

    # Calculate BLEU score for a subset of examples
    bleu_scores = []
    smooth = SmoothingFunction().method1

    for i in range(min(100, len(decoded_preds))):
        if not decoded_preds[i] or not decoded_labels[i]:
            bleu_scores.append(0.0)
            continue

        reference = [decoded_labels[i].split()]
        candidate = decoded_preds[i].split()
        try:
            bleu_score = sentence_bleu(reference, candidate, smoothing_function=smooth)
            bleu_scores.append(bleu_score)
        except Exception:
            bleu_scores.append(0.0)

    # Display a few examples (safely)
    num_examples = min(3, len(decoded_preds))
    for i in range(num_examples):
        # Fix: Use rougeL_scores instead of rouge_scores
        if i < len(rougeL_scores):
            print(f"\nExample {i+1}:")
            print(f"Predicted: {decoded_preds[i][:100]}...")
            print(f"Actual: {decoded_labels[i][:100]}...")
            print(f"ROUGE-L: {rougeL_scores[i]:.4f}")

    # Ensure we have scores to calculate metrics
    if not rouge1_scores:
        rouge1_scores = [0.0]
    if not rouge2_scores:
        rouge2_scores = [0.0]
    if not rougeL_scores:
        rougeL_scores = [0.0]
    if not bleu_scores:
        bleu_scores = [0.0]

    # Combine metrics
    results = {
        'rouge1': sum(rouge1_scores) / len(rouge1_scores),
        'rouge2': sum(rouge2_scores) / len(rouge2_scores),
        'rougeL': sum(rougeL_scores) / len(rougeL_scores),
        'bleu': sum(bleu_scores) / len(bleu_scores),
    }

    return results

# Custom prediction generation function
def generate_predictions(model, dataset, tokenizer, batch_size=8, max_length=128):
    """Generate predictions from the model for the dataset"""
    predictions = []
    model.eval()

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating predictions"):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            # Generate predictions
            outputs = model.generate(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                max_length=max_length,
                num_beams=4,
                length_penalty=1.0,
                early_stopping=True
            )

            # Decode predictions
            decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(decoded_outputs)

    return predictions

### Model Training

In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./clinical_reasoning_model",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    weight_decay=0.01,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=10,
    report_to="none",  # Disable wandb, tensorboard, etc.
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train model
print("Starting training...")
train_start_time = time.time()
trainer.train()
train_duration = time.time() - train_start_time
print(f"Training completed in {train_duration / 60:.2f} minutes")

# Evaluate model
eval_results = trainer.evaluate()
print("\nEvaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# Save the model
model_save_path = "./clinical_reasoning_final_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

Starting training...


Epoch,Training Loss,Validation Loss
